<a href="https://colab.research.google.com/github/kasamradhika014-crypto/Ai-resume-screening-system/blob/main/JD_Resume_Matcher_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Sample Resume & Job Description to test our math and logic
resume_text = """
B.Sc. final year student in Mathematics, AI, and Computer Science.
Skills: Python, SQL, SQLite, FastAPI, Streamlit, Pandas, NumPy, Scikit-learn, Git, HTML, CSS.
Projects: Built a voice-enabled digital storefront using Python, FastAPI, Gemini API, and SQLite.
Completed data analytics internship analyzing retail transactions using Python and SQL.
"""

jd_text = """
Looking for a Junior AI & Python Developer in Hyderabad.
Required Skills: Python, REST APIs, FastAPI, SQL, Docker, Git, Machine Learning, LLM APIs.
Responsibilities: Build backend APIs, integrate Generative AI models, and query relational databases.
"""

def clean_text(text: str) -> str:
    """Lowercase text and remove special punctuation while keeping tech terms."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\+\#]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

print("Cleaned Resume:", clean_text(resume_text)[:90], "...")

Cleaned Resume: b sc final year student in mathematics ai and computer science skills python sql sqlite fa ...


In [10]:
def compute_ats_similarity(resume: str, jd: str) -> float:
    cleaned_docs = [clean_text(resume), clean_text(jd)]

    # Convert text into TF-IDF vectors (ignoring common English stop words)
    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
    tfidf_matrix = vectorizer.fit_transform(cleaned_docs)

    # Compute cosine similarity between vector 0 (resume) and vector 1 (JD)
    score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
    return round(score * 100, 2)

ats_score = compute_ats_similarity(resume_text, jd_text)
print(f"ATS TF-IDF Cosine Similarity Score: {ats_score}%")

ATS TF-IDF Cosine Similarity Score: 9.97%


In [11]:
# Curated technical skill taxonomy (stored as a Python set for O(1) lookup)
TECH_SKILLS = {
    "python", "c", "c++", "java", "sql", "sqlite", "fastapi", "streamlit",
    "pandas", "numpy", "scikit-learn", "git", "html", "css", "javascript",
    "docker", "rest apis", "machine learning", "deep learning", "llm apis",
    "generative ai", "gemini api", "data structures", "algorithms"
}

def analyze_skill_gap(resume: str, jd: str, skill_vocab: set) -> dict:
    res_clean = f" {clean_text(resume)} "
    jd_clean = f" {clean_text(jd)} "

    # Extract skills present in JD and Resume
    jd_required = {skill for skill in skill_vocab if f" {skill} " in jd_clean}
    resume_has = {skill for skill in skill_vocab if f" {skill} " in res_clean}

    # Set operations: Intersection (&) for matched, Difference (-) for missing
    matched = sorted(jd_required & resume_has)
    missing = sorted(jd_required - resume_has)

    match_pct = round((len(matched) / len(jd_required)) * 100, 2) if jd_required else 0.0

    return {
        "skill_match_pct": match_pct,
        "matched_skills": matched,
        "missing_skills": missing
    }

gap_report = analyze_skill_gap(resume_text, jd_text, TECH_SKILLS)
print("Skill Match %:", gap_report["skill_match_pct"], "%")
print("Matched Skills:", gap_report["matched_skills"])
print("Missing Skills:", gap_report["missing_skills"])

Skill Match %: 44.44 %
Matched Skills: ['fastapi', 'git', 'python', 'sql']
Missing Skills: ['docker', 'generative ai', 'llm apis', 'machine learning', 'rest apis']


In [12]:
import json
import time
from google import genai
from google.genai import types
from google.colab import userdata
from pydantic import BaseModel, Field

class ResumeEvaluation(BaseModel):
    semantic_fit_score: int = Field(description="Holistic fit score from 0 to 100 based on conceptual skill overlap.")
    transferable_skills_found: list[str] = Field(description="JD skills satisfied indirectly by the candidate's projects.")
    true_skill_gaps: list[str] = Field(description="Skills genuinely absent from the resume that the candidate should learn.")
    rewritten_resume_bullets: list[str] = Field(description="Two ATS-optimized resume bullet points tailored to this JD.")

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

prompt = f"""
You are a senior technical recruiter evaluating a B.Sc. (AI/ML) candidate's resume against a Job Description.
Rule-based keyword matcher found:
- Matched Keywords: {gap_report['matched_skills']}
- Exact-Match Gaps: {gap_report['missing_skills']}

Evaluate semantic overlap fairly (for example, Gemini API satisfies LLM APIs/Generative AI, FastAPI satisfies REST APIs, Scikit-learn satisfies Machine Learning).

RESUME:
{resume_text}

JOB DESCRIPTION:
{jd_text}
"""

def generate_with_fallback(prompt_text: str) -> dict:
    # Try primary model first, then fast low-latency fallback models if 503 occurs
    candidate_models = ["gemini-3.8-flash", "gemini-3.5-flash-lite", "gemini-3.6-flash", "gemini-4-hydrogen"]

    for model_name in candidate_models:
        try:
            print(f"Trying {model_name}...")
            res = client.models.generate_content(
                model=model_name,
                contents=prompt_text,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=ResumeEvaluation,
                    temperature=0.2,
                ),
            )
            print(f"Success using {model_name}!")
            return json.loads(res.text)
        except Exception as e:
            print(f"{model_name} busy/unavailable ({type(e).__name__}), switching fallback...")
            time.sleep(1)
    raise RuntimeError("All candidate models are temporarily busy. Wait 30 seconds and re-run.")

ai_report = generate_with_fallback(prompt)
print(json.dumps(ai_report, indent=2))

Trying gemini-3.8-flash...
gemini-3.8-flash busy/unavailable (ServerError), switching fallback...
Trying gemini-3.5-flash-lite...
Success using gemini-3.5-flash-lite!
{
  "semantic_fit_score": 88,
  "transferable_skills_found": [
    "REST APIs",
    "LLM APIs",
    "Generative AI",
    "Machine Learning"
  ],
  "true_skill_gaps": [
    "Docker"
  ],
  "rewritten_resume_bullets": [
    "Developed and deployed high-performance REST APIs using FastAPI and Python to power voice-enabled digital storefronts.",
    "Integrated Gemini API and applied Scikit-learn machine learning techniques to process data and deliver generative AI capabilities."
  ]
}


In [13]:
def evaluate_candidate(resume: str, jd: str) -> dict:
    # Layer 1: Deterministic TF-IDF Cosine Similarity (20% weight)
    tfidf_score = compute_ats_similarity(resume, jd)

    # Layer 2: DSA Set-Based Exact Skill Match (30% weight)
    gap = analyze_skill_gap(resume, jd, TECH_SKILLS)
    exact_score = gap["skill_match_pct"]

    # Layer 3: Generative AI Semantic Evaluation with Model Fallback (50% weight)
    eval_prompt = f"""
    You are a senior technical recruiter evaluating a B.Sc. (AI/ML) candidate's resume against a Job Description.
    Rule-based keyword matcher found:
    - Matched Keywords: {gap['matched_skills']}
    - Exact-Match Gaps: {gap['missing_skills']}
    Evaluate semantic overlap fairly (recognizing transferable tech stacks).
    RESUME: {resume}
    JOB DESCRIPTION: {jd}
    """
    ai_data = generate_with_fallback(eval_prompt)
    semantic_score = ai_data["semantic_fit_score"]

    # Weighted Hybrid Final Score
    final_score = round((0.20 * tfidf_score) + (0.30 * exact_score) + (0.50 * semantic_score), 2)

    return {
        "final_hybrid_score": final_score,
        "score_breakdown": {
            "tfidf_cosine_pct": tfidf_score,
            "exact_skill_match_pct": exact_score,
            "ai_semantic_fit_pct": semantic_score,
        },
        "matched_skills": gap["matched_skills"],
        "transferable_skills_found": ai_data["transferable_skills_found"],
        "true_skill_gaps": ai_data["true_skill_gaps"],
        "tailored_resume_bullets": ai_data["rewritten_resume_bullets"],
    }

# Test the full pipeline end-to-end!
final_report = evaluate_candidate(resume_text, jd_text)
print(json.dumps(final_report, indent=2))

Trying gemini-3.8-flash...
gemini-3.8-flash busy/unavailable (ServerError), switching fallback...
Trying gemini-3.5-flash-lite...
Success using gemini-3.5-flash-lite!
{
  "final_hybrid_score": 57.83,
  "score_breakdown": {
    "tfidf_cosine_pct": 9.97,
    "exact_skill_match_pct": 44.44,
    "ai_semantic_fit_pct": 85
  },
  "matched_skills": [
    "fastapi",
    "git",
    "python",
    "sql"
  ],
  "transferable_skills_found": [
    "LLM APIs",
    "Generative AI",
    "REST APIs"
  ],
  "true_skill_gaps": [
    "Docker",
    "Machine Learning"
  ],
  "tailored_resume_bullets": [
    "Engineered and deployed scalable REST APIs using FastAPI and Python to power voice-enabled digital storefront features.",
    "Integrated LLM APIs and Generative AI models into backend services while managing relational databases with SQL and SQLite."
  ]
}
